## 4. LSTM Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src.dataloader import load_cgm_stats, load_tensors, load_participant_ids
from src import metrics
from src import train
from src.models.nn_baseline import LSTMPredictor, HORIZONS

MODEL_NAME = 'LSTM'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'lstm'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'lstm_stability_summary.json'

In [3]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

test_participant_ids = load_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print(f'number of train batches: ', len(train_loader), '\nnumber of valuation batches: ', len(val_loader),
      '\nnumber of test batches: ', len(test_loader))
print('number of test participants:', len(PARTICIPANT_IDS))
print('Corresponding participants ID: \n', PARTICIPANT_IDS)

sample_cgm, sample_target = next(iter(train_loader))
print('\ninput batch shape (cgm):', tuple(sample_cgm.shape))
print('target batch shape:', tuple(sample_target.shape))

number of train batches:  327 
number of valuation batches:  66 
number of test batches:  66
number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

input batch shape (cgm): (256, 24)
target batch shape: (256, 5)


In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory -- also written to disk individually below

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)  # before constructing the model -- weight init draws from this
    model = LSTMPredictor()

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'lstm' /f'lstm_seed{seed}',
        verbose=False,  # quiet per-epoch text logging -- the progress bar below (and the one
                         # summary line per seed, after) carry that job instead
        progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader)
    y_pred_test, y_true_test = train.predict(model, test_loader)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    # Same test predictions, broken down per participant instead of pooled -- this is what lets
    # the aggregation cells separate "variability across seeds" from "variability across
    # participants" instead of only ever reporting the former.
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'lstm_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    print(f"\n{MODEL_NAME} | seed={seed} | stopped at epoch {n_epochs}/50")

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")



Test results — LSTM | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: LSTMPredictor  params: 207,749


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]


LSTM | seed=7 | stopped at epoch 19/50
 15 min | RMSE 11.30 | MAE 7.61 | gRMSE 12.17 | Zone A 97.03% | A+B 99.83%
 30 min | RMSE 18.52 | MAE 12.22 | gRMSE 21.18 | Zone A 90.48% | A+B 99.55%
 60 min | RMSE 27.31 | MAE 18.01 | gRMSE 33.20 | Zone A 81.88% | A+B 98.95%
 90 min | RMSE 32.66 | MAE 21.70 | gRMSE 40.65 | Zone A 75.73% | A+B 98.40%
120 min | RMSE 35.82 | MAE 24.06 | gRMSE 45.06 | Zone A 71.51% | A+B 98.06%

--------------------------------------------------------------------------------
None

Test results — LSTM | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: LSTMPredictor  params: 207,749


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]


LSTM | seed=14 | stopped at epoch 23/50
 15 min | RMSE 11.36 | MAE 7.58 | gRMSE 12.64 | Zone A 97.12% | A+B 99.83%
 30 min | RMSE 18.60 | MAE 12.21 | gRMSE 21.52 | Zone A 90.64% | A+B 99.54%
 60 min | RMSE 27.18 | MAE 18.03 | gRMSE 32.93 | Zone A 81.59% | A+B 98.98%
 90 min | RMSE 32.28 | MAE 21.73 | gRMSE 39.95 | Zone A 75.69% | A+B 98.62%
120 min | RMSE 35.23 | MAE 23.98 | gRMSE 44.01 | Zone A 71.38% | A+B 98.43%

--------------------------------------------------------------------------------
None

Test results — LSTM | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: LSTMPredictor  params: 207,749


...


LSTM | seed=56 | stopped at epoch 29/50
 15 min | RMSE 11.18 | MAE 7.58 | gRMSE 12.30 | Zone A 97.01% | A+B 99.83%
 30 min | RMSE 18.43 | MAE 12.23 | gRMSE 21.30 | Zone A 90.52% | A+B 99.58%
 60 min | RMSE 27.12 | MAE 18.06 | gRMSE 33.01 | Zone A 81.78% | A+B 98.95%
 90 min | RMSE 32.32 | MAE 21.84 | gRMSE 40.09 | Zone A 75.55% | A+B 98.43%
120 min | RMSE 35.37 | MAE 24.23 | gRMSE 44.25 | Zone A 70.78% | A+B 98.16%

--------------------------------------------------------------------------------
None

Test results — LSTM | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: LSTMPredictor  params: 207,749


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]


LSTM | seed=63 | stopped at epoch 27/50
 15 min | RMSE 11.15 | MAE 7.59 | gRMSE 12.27 | Zone A 97.05% | A+B 99.81%
 30 min | RMSE 18.42 | MAE 12.27 | gRMSE 21.31 | Zone A 90.54% | A+B 99.59%
 60 min | RMSE 27.14 | MAE 18.10 | gRMSE 33.00 | Zone A 81.64% | A+B 98.97%
 90 min | RMSE 32.35 | MAE 21.76 | gRMSE 40.15 | Zone A 75.58% | A+B 98.46%
120 min | RMSE 35.40 | MAE 24.01 | gRMSE 44.41 | Zone A 71.26% | A+B 98.11%

--------------------------------------------------------------------------------
None

Test results — LSTM | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: LSTMPredictor  params: 207,749


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]


LSTM | seed=70 | stopped at epoch 26/50
 15 min | RMSE 11.16 | MAE 7.50 | gRMSE 12.24 | Zone A 97.15% | A+B 99.88%
 30 min | RMSE 18.42 | MAE 12.13 | gRMSE 21.11 | Zone A 90.67% | A+B 99.60%
 60 min | RMSE 26.96 | MAE 17.80 | gRMSE 32.51 | Zone A 81.89% | A+B 99.04%
 90 min | RMSE 32.06 | MAE 21.51 | gRMSE 39.53 | Zone A 75.94% | A+B 98.47%
120 min | RMSE 35.12 | MAE 23.78 | gRMSE 43.80 | Zone A 71.71% | A+B 98.18%

--------------------------------------------------------------------------------
None


In [6]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [
                r['test_results'][h]['ceg']['A']
            for r in all_results
            ]
        elif m == 'zone_ab':
            seed_values = [
                r['test_results'][h]['ceg']['A']
                + r['test_results'][h]['ceg']['B']
                for r in all_results
            ]
        else:
            seed_values = [
                r['test_results'][h][m]
                for r in all_results
            ]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [
                _extract(r['test_results_per_participant'][pid][h], m)
                for r in all_results
            ]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {
            'mean': mean,
            'sd1_across_seeds': sd1,
            'sd2_across_participants': sd2,
        }

# ---------------------------------------------------------
# Print: mean +/- sd1 (sd2)
# ---------------------------------------------------------
print(
    f'=== {MODEL_NAME} test set: '
    f'overall mean +/- SD_seed (SD_participant) ==='
)

for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- "
        f"{s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- "
        f"{s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- "
        f"{s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- "
        f"{s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
        f"{s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

=== LSTM test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.29 +/- 0.11 (2.00) mg/dL   MAE = 7.62 +/- 0.07 (1.51)   gRMSE = 12.46 +/- 0.24 (3.12)   Zone A = 97.08 +/- 0.09 (1.91)%   Zone A+B = 99.83 +/- 0.03 (0.24)%
 30-min:  RMSE = 18.57 +/- 0.13 (3.94) mg/dL   MAE = 12.28 +/- 0.09 (2.64)   gRMSE = 21.49 +/- 0.29 (6.44)   Zone A = 90.53 +/- 0.15 (2.98)%   Zone A+B = 99.56 +/- 0.04 (0.53)%
 60-min:  RMSE = 27.37 +/- 0.30 (9.00) mg/dL   MAE = 18.06 +/- 0.19 (5.51)   gRMSE = 33.34 +/- 0.56 (14.08)   Zone A = 81.67 +/- 0.21 (4.77)%   Zone A+B = 98.95 +/- 0.06 (1.07)%
 90-min:  RMSE = 32.63 +/- 0.43 (12.71) mg/dL   MAE = 21.77 +/- 0.24 (7.69)   gRMSE = 40.61 +/- 0.74 (19.75)   Zone A = 75.55 +/- 0.48 (6.42)%   Zone A+B = 98.38 +/- 0.15 (1.65)%
120-min:  RMSE = 35.68 +/- 0.46 (15.10) mg/dL   MAE = 24.06 +/- 0.25 (9.33)   gRMSE = 44.84 +/- 0.78 (23.41)   Zone A = 71.41 +/- 0.55 (6.81)%   Zone A+B = 98.05 +/- 0.22 (2.33)%


In [7]:
# Save one final summary JSON
with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/lstm/lstm_seed*_results.json
saved aggregate summary to /content/drive/MyDrive/dissertation/experiments/1_baseline/results/lstm/lstm_stability_summary.json
